In [ ]:
import pandas as pd

from scipy import stats

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data = pd.read_csv(r'E:\IDE\PY_10_Введение в Pandas\ds_salaries\ds_salaries.csv')
print(data.shape)
display(data.head())
display(data.tail())

In [ ]:
data.info()

In [ ]:
print('В датасете {} записей без пропущенных значений.'.format(data.shape[0]))


In [ ]:
print('Количество дубликатов: {}'.format(data[data.duplicated()].shape[0]))


In [ ]:
data.drop('Unnamed: 0', axis = 1, inplace = True)
data

In [ ]:
data.info()
data.corr(numeric_only = True)

In [ ]:
sns.heatmap(data.corr(numeric_only=True), annot=True)

Сильной корреляции между признаками нет.

In [ ]:
import numpy as np

data_numeric = data.select_dtypes(include=[np.number])
numeric_cols = data_numeric.columns.values
print(numeric_cols)

In [ ]:
data_non_numeric = data.select_dtypes(exclude=[np.number])
non_numeric_cols = data_non_numeric.columns.values
print(non_numeric_cols)

Имеем признаки:

категориальные:

- experience_level
- employment_type
- remote_ratio
- company_size
- company_location
- salary_currency
- employee_residence
- job_title

числовые:

- work_year
- salary
- salary_in_usd

In [ ]:
data.boxplot(column=['work_year'])

Выбросов нет

In [ ]:
data.boxplot(column=['salary'])


Выброс есть

In [ ]:
data.boxplot(column=['salary_in_usd'])


Выброс есть

In [ ]:
data.boxplot(column=['remote_ratio'])

Выбросов нет

In [ ]:
counts = data['experience_level'].value_counts().reset_index()
counts.columns = ['category', 'count']
print(counts)

In [ ]:
counts_employment_type = data['employment_type'].value_counts().reset_index()
counts_employment_type.columns = ['category', 'count']
print(counts_employment_type)

In [ ]:
counts = data['company_size'].value_counts().reset_index()
counts.columns = ['category', 'count']
print(counts)

1. Как соотносятся зарплаты Data Scientist и Data Engineer в 2022 году?

In [ ]:
year22_ds_de = data[(data['work_year'] == 2022) & 
                    (data['job_title'].isin(['Data Scientist', 'Data Engineer']))].copy()

medians_2022 = year22_ds_de.groupby('job_title')['salary_in_usd'].median().reset_index()
medians_2022.columns = ['job_title', 'медианная зарплата']

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
fig.subplots_adjust(hspace=0.3, wspace=0.3)

sns.barplot(data=medians_2022, x='job_title', y='медианная зарплата', ax=axes[0])
axes[0].set_xlabel("должность")
axes[0].set_ylabel("медианная зарплата (USD)")
axes[0].set_title('Медианные зарплаты за 2022г')

sns.boxplot(data=year22_ds_de, x='job_title', y='salary_in_usd', ax=axes[1])
axes[1].set_xlabel("должность")
axes[1].set_ylabel("зарплата (USD)")
axes[1].set_title('Распределение зарплат DS и DE за 2022г')

plt.show()

Вывод: медианная заработная плата Дата саентиста выше чем заработная плата Дата инженера за 2022г

2. Наблюдается ли ежегодный рост зарплат у специалистов Data Scientist?

In [ ]:
data_sal = data.groupby( ['work_year'], observed=False )['salary_in_usd'].count().to_frame()
data_sal.columns = ['Количество']

data_median = data.groupby( ['work_year'], observed=False )['salary_in_usd'].median().round(0).to_frame()
data_median.columns = ['Медианная зарплата']

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

sns.barplot(data_sal, x='work_year',y='Количество', ax=axes[0])
axes[0].set_title("Количество по годам")

sns.barplot(data_median, x='work_year',y='Медианная зарплата', ax=axes[1])
axes[1].set_title("Зарплата по годам")

sns.boxplot(data, x='salary_in_usd', y='work_year', orient='h', ax=axes[2])
axes[2].set( ylabel="Год", xlabel="Зарплата")
axes[2].set_title('Распределение зарплаты')

Выводы: 
1. Прирост Зп по годам, особенно сильный скачок в 2022
2. Медианная ЗП отличается по годам, наибольший прирост в 2022
3. Выбросы имеются во всех анализируемых годах
4. Медианные ЗП в 2020 и 2021 примерно равно

In [ ]:
data_median = data.groupby('company_size')['salary_in_usd'].median().round().to_frame()
data_median.columns = ['Медианная зарплата']

data_ds = data[(data['job_title'] == 'Data Scientist')].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
sns.barplot(data=data_median, x='company_size', y='Медианная зарплата', ax=axes[0])
axes[0].set_xlabel("размер компании")
axes[0].set_ylabel("медианная зарплата (USD)")
axes[0].set_title('Соотношение размера компании и зарплат')

sns.boxplot(data=data_ds, x='company_size', y='salary_in_usd', ax=axes[1])
axes[1].set_xlabel("размер компании")
axes[1].set_ylabel("зарплата (USD)")
axes[1].set_title('Соотношение размера компании и зарплат')

plt.show()

Вывод: медианные зарплаты в компаниях разного размера отличаются, причем распределение незакономерно наибольшая медианная заробтная плата в компании среднего размера, далее по убыванию в компании крупного размера и далее в маленькой компании.

4. Есть ли связь между наличием должностей Data Scientist и Data Engineer и размером компании?

In [ ]:
cross_tab = pd.crosstab(index=data['company_size'], columns=data['job_title'])
cross_tab

In [ ]:
ds_de = data[(data['job_title'].isin(['Data Scientist', 'Data Engineer']))].copy()

cross_tab = pd.crosstab(index=ds_de['company_size'], columns=ds_de['job_title'])

ax = sns.heatmap(cross_tab, annot=True)
ax.set(xlabel='Должность', ylabel='Размер компании')
plt.title('Таблица сопряженности')
plt.show()

Выводы: Связь между наличием должностей Data Scientist и Data Engineer и размером компании имеется. В компаниях среднего размера больше количество DI, однако в крупных и мелких компаниях в штате бльше DS.

КАКИЕ ФАКТОРЫ ВЛИЯЮТ НА ЗАРПЛАТУ DATA SCIENTIST?

In [ ]:
alpha = 0.05
print("Уровень значимости равен = {:.2f}".format(alpha))

1. Зарплата Data Scientist зависит от года?

Нулевая гипотеза(Н0): ЗП DS не зависит от года

Альтернативная гипотеза(Н1): ЗП DS зависит от года

In [ ]:
def decision_normality(p):
    print('p-value = {:.3f}'.format(p))

    if p <= alpha:
         print('p-значение меньше, чем заданный уровень значимости {:.2f}. Распределение отлично от нормального'.format(alpha))
    else:
         print('p-значение больше, чем заданный уровень значимости {:.2f}. Распределение является нормальным'.format(alpha))

def decision_hypothesis(p):
    print('p-value = {:.3f}'.format(p))
    if p <= alpha:
        print('p-значение меньше, чем заданный уровень значимости {:.2f}. Отвергаем нулевую гипотезу в пользу альтернативной.'.format(alpha))
    else:
        print('p-значение больше, чем заданный уровень значимости {:.2f}. У нас нет оснований отвергнуть нулевую гипотезу.'.format(alpha))

In [ ]:
salary_2020 = data_ds.loc[data_ds['work_year']==2020, 'salary_in_usd'].copy()
salary_2021 = data_ds.loc[data_ds['work_year']==2021, 'salary_in_usd'].copy()
salary_2022 = data_ds.loc[data_ds['work_year']==2022, 'salary_in_usd'].copy()

print('2020: ')
result = stats.shapiro(salary_2020)
decision_normality(result[1])

print('2021:')
result = stats.shapiro(salary_2021)
decision_normality(result[1])

print('2022:')
result = stats.shapiro(salary_2022)
decision_normality(result[1])

Сравниваем 3 независимых группы с количественными признаками. Распределение отлично от нормального. Следовательно,мы должны использовать тест Краскела-Уоллиса.

In [ ]:
_, p = stats.kruskal(salary_2022, salary_2021, salary_2020)
decision_hypothesis(p)

Вывод: заработные платы Data Scientist отличаются по годам

In [ ]:
_, p = stats.kruskal(salary_2022, salary_2021, salary_2020)
decision_hypothesis(p)

2. На зарплату Data Scientist влияет размер компании?

Нулевая гипотеза(Н0): на ЗП DS не влияет размер компании

Альтернативная гипотеза(Н1): на ЗП DS влияет размер компании

In [ ]:
salary_s = data_ds.loc[data_ds['company_size']=='S', 'salary_in_usd']
salary_m = data_ds.loc[data_ds['company_size']=='M', 'salary_in_usd']
salary_l = data_ds.loc[data_ds['company_size']=='L', 'salary_in_usd']

print('S или небольшая компания:')
result = stats.shapiro(salary_s)
decision_normality(result[1])

print('M или средняя компания:')
result = stats.shapiro(salary_m)
decision_normality(result[1])

print('L или крупная компания:')
result = stats.shapiro(salary_l)
decision_normality(result[1])

Сравниваем 3 независимых группы с количественными признаками. Распределение отлично от нормального. Следовательно,мы должны использовать тест Краскела-Уоллиса.

In [ ]:
_, p = stats.kruskal(salary_s, salary_m, salary_l)
decision_hypothesis(p)

Вывод: размер компании влияет на заработную плату Data Scientist

3. Зарплата Data Scientist в компании среднего размера и небольшой компании различны

Альтернативная гипотеза(Н1): зарплата Data Scientist в компании среднего размера и небольшой компании различны

Нулевая гипотеза(Н0): зарплата Data Scientist в компании среднего размера и небольшой компании не отличается

In [ ]:
print('S или небольшая компания:')
result = stats.shapiro(salary_s)
decision_normality(result[1])

print('M или средняя компания:')
result = stats.shapiro(salary_m)
decision_normality(result[1])

Сравниваем 2 независимых группы с количественными признаками. Распределение является нормальным. Следовательно,мы должны использовать U-критерий Манна-Уитни.


In [ ]:
_, p = stats.mannwhitneyu(salary_s, salary_m, alternative='two-sided')
decision_hypothesis(p)


Вывод: зарплата Data Scientist в компании среднего размера и небольшой компании различны


4. Медианная ЗП Data Scientist в компании среднего размера больше 100 000 долларов?

Нулевая гипотеза (Н0): медианная зарплата в компании среднего небольшого размера меньше или равна 100 тыс долларов

Альтернативная гипотеза (Н1): медианная зарплата в компании среднего размера больше 100 тыс долларов


In [ ]:
print('M или средняя компания:')
result = stats.shapiro(salary_m)
decision_normality(result[1])

1 группа с количественными признаками. Распределение является нормальным. Следовательно, мы должны использовать Одновыборочный t-критерий.

In [ ]:
_, p = stats.ttest_1samp(salary_m, popmean=100000, alternative='greater')
decision_hypothesis(p)

Вывод: медианная заработная плата в компании среднего размера больше чем 100 000 долларов

5. Есть ли связь между наличием должностей Data Scientist и Data Engineer и размером компании?

Нулевая гипотеза (Н0): связи нет - признаки независимы

Альтернативная гипотеза (Н1): связь есть - признаки зависимы

2 группы с независимыми категориальными признаками. Следовательно, мы должны использовать хи-квадрат.


In [ ]:
cross_table = pd.crosstab(ds_de['job_title'], ds_de['company_size'])
cross_table

In [ ]:
_, p, _, _ = stats.chi2_contingency(cross_table)
decision_hypothesis(p)

Вывод: связь между наличием должностей Data Scientist и Data Engineer и размером компании есть


6. Как соотносятся зарплаты Data Scientist и Data Engineer в 2022 году?

Нулевая гипотеза (H0): ЗП DS равна или меньше, чем у DE в 2022

Альтернативная гипотеза (Н1):ЗП DS больше, чем у DE в 2022

In [ ]:
data_de = data[(data['job_title'] == 'Data Engineer')].copy()

salary_ds_22 = data_ds.loc[ds_de['work_year']==2022, 'salary_in_usd'].copy()
salary_de_22 = data_de.loc[ds_de['work_year']==2022, 'salary_in_usd'].copy()

print('Для 2022, Data Science:')
result = stats.shapiro(salary_ds_22)
decision_normality(result[1])

print('Для 2022, Data Engeneer:')
result = stats.shapiro(salary_de_22)
decision_normality(result[1])

2 группы с независимыми количественными признаками. Следовательно, мы должны использовать U-критерий Манна-Уитни.


In [ ]:
_, p = stats.mannwhitneyu(salary_ds_22, salary_de_22, alternative='greater')
decision_hypothesis(p)

Итоговые результаты исследования по данным о зарплатах в сфере Data Science за 2020–2022 годы:
- наблюдается рост заработной платы по годам, особенно за 2022 год
- заработные платы в 2021 году и 2020 году примерно равны
- медианные заработные платы в компаниях разного размера отличаются, причем незакономерно наибольшая зарплата в компании среднего размера, наименьшая в компании малого размера
- размер компании влияет на размер заплаты дата саентиста
- медианная зарплата Дата саентиста в компании среднего размера больше 100 000 долларов